# 🧩 Mini-Lab: Basic Document Extraction

**Module 3: Structured Extraction** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Extract** structured data from unstructured text using zero-shot prompting
2. **Improve** extraction accuracy with few-shot examples
3. **Parse** LLM output into Python objects
4. **Compare** zero-shot vs few-shot performance

## Target Concepts

| Concept | Description |
|---------|-------------|
| Zero-shot extraction | Extracting data without providing examples, relying on model's general knowledge |
| Few-shot learning | Providing examples to guide the LLM toward consistent output format |
| Structured output | Converting unstructured text into structured data formats like JSON |
| Output parsing | Converting LLM string responses into typed Python objects |

In [6]:
import json
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

load_dotenv()
client = OpenAI()

# Data path - relative to project folder
DATA_DIR = Path('../data')

print('✓ Setup complete')
print(f'✓ Data directory: {DATA_DIR}')

✓ Setup complete
✓ Data directory: ..\data


---
## 1. Load Sample Documents

Let's start by loading some invoice documents to practice extraction.

In [7]:
# Load all invoices
INVOICES = {}
invoice_dir = DATA_DIR / 'invoices'

for file in sorted(invoice_dir.glob('*.txt')):
    INVOICES[file.stem] = file.read_text(encoding='utf-8')

print(f'✓ Loaded {len(INVOICES)} invoices:')
for name in INVOICES.keys():
    print(f'  - {name}')

# Preview the first invoice
sample_name = list(INVOICES.keys())[0]
sample_invoice = INVOICES[sample_name]

print(f'\n📄 Preview of {sample_name}:')
print('=' * 60)
print(sample_invoice[:800] + '...' if len(sample_invoice) > 800 else sample_invoice)

✓ Loaded 6 invoices:
  - invoice_001_corporate
  - invoice_002_simple_receipt
  - invoice_003_email_style
  - invoice_004_european_format
  - invoice_005_handwritten_style
  - invoice_006_minimal

📄 Preview of invoice_001_corporate:
TechCorp Solutions Inc. | 1234 Innovation Drive, Suite 500 | San Francisco, CA 94105 | Phone: (415) 555-0123 | billing@techcorpsolutions.com

═══════════════════════════════════════════════════════════════════════════════
                                    INVOICE
═══════════════════════════════════════════════════════════════════════════════

Invoice #: INV-2024-0842
Issue Date: 2024-01-15
Payment Due: 2024-02-14
Terms: Net 30 days

CLIENT INFORMATION:
Acme Corporation
5678 Business Park Boulevard
Los Angeles, CA 90012

═══════════════════════════════════════════════════════════════════════════════
                                    DETAILS
═══════════════════════════════════════════════════════════════════════════════

#001 | Cloud Infrastructure Setup 

---
## 2. Zero-Shot Extraction

In **zero-shot extraction**, we simply ask the LLM to extract data without providing any examples.
This is the simplest approach but may produce inconsistent output formats.

In [8]:
def zero_shot_extract(document: str, fields: list[str]) -> str:
    """
    Extract fields from a document using zero-shot prompting.
    
    Args:
        document: The document text
        fields: List of field names to extract
        
    Returns:
        Raw LLM response (string)
    """
    prompt = f'''Extract the following fields from the document: {fields}

Document:
{document}

Return the extracted data as JSON.'''
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0  # Low temperature for consistent extraction
    )
    
    return response.choices[0].message.content

# Test zero-shot extraction
fields_to_extract = ['invoice_number', 'vendor_name', 'customer_name', 'total']
result = zero_shot_extract(sample_invoice, fields_to_extract)

print('📤 Zero-shot extraction result:')
print(result)

📤 Zero-shot extraction result:
```json
{
  "invoice_number": "INV-2024-0842",
  "vendor_name": "TechCorp Solutions Inc.",
  "customer_name": "Acme Corporation",
  "total": "$24,686.46"
}
```


### Observations

Notice that zero-shot extraction:
- ✅ Works reasonably well for simple documents
- ⚠️ Output format may vary (sometimes markdown, sometimes plain JSON)
- ⚠️ Field names may not match exactly what we requested
- ❌ May include extra explanation text

---
## 3. Zero-Shot with System Prompt

Adding a **system prompt** helps define the extractor's behavior more clearly.

In [9]:
def zero_shot_with_system(document: str, fields: list[str]) -> str:
    """
    Extract fields using a system prompt for better control.
    """
    system_prompt = '''You are a document data extraction assistant.
Your task is to extract specific fields from documents and return them as JSON.

Rules:
- Return ONLY valid JSON, no markdown or explanation
- Use null for fields that cannot be found
- Use exact field names as requested
- Be precise and accurate'''

    user_prompt = f'''Extract these fields: {fields}

Document:
{document}'''
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        temperature=0
    )
    
    return response.choices[0].message.content

# Test with system prompt
result = zero_shot_with_system(sample_invoice, fields_to_extract)

print('📤 Zero-shot with system prompt:')
print(result)

# Try to parse as JSON
try:
    parsed = json.loads(result)
    print('\n✅ Valid JSON! Parsed result:')
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f'\n❌ Invalid JSON: {e}')

📤 Zero-shot with system prompt:
{
  "invoice_number": "INV-2024-0842",
  "vendor_name": "TechCorp Solutions Inc.",
  "customer_name": "Acme Corporation",
  "total": "$24,686.46"
}

✅ Valid JSON! Parsed result:
{
  "invoice_number": "INV-2024-0842",
  "vendor_name": "TechCorp Solutions Inc.",
  "customer_name": "Acme Corporation",
  "total": "$24,686.46"
}


---
## 4. Few-Shot Extraction

**Few-shot learning** provides examples to guide the LLM. This significantly improves:
- Output format consistency
- Field naming accuracy
- Handling of edge cases

In [10]:
# Define few-shot examples
FEW_SHOT_EXAMPLES = [
    {
        'document': '''Invoice #A1234
From: TechCo Inc.
To: Customer ABC
Date: Jan 1, 2024
Items: Widget x2 = $100
Total: $100''',
        'extracted': {
            'invoice_number': 'A1234',
            'vendor_name': 'TechCo Inc.',
            'customer_name': 'Customer ABC',
            'total': 100.0
        }
    },
    {
        'document': '''INVOICE
Number: INV-2024-99
Seller: MegaCorp LLC
Buyer: SmallBiz Co.
Services: Consulting - $5,000
Tax: $400
Amount Due: $5,400''',
        'extracted': {
            'invoice_number': 'INV-2024-99',
            'vendor_name': 'MegaCorp LLC',
            'customer_name': 'SmallBiz Co.',
            'total': 5400.0
        }
    }
]

print('📚 Few-shot examples defined:')
for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
    print(f'\nExample {i}:')
    print(f'  Document preview: {ex["document"][:50]}...')
    print(f'  Extracted: {ex["extracted"]}')

📚 Few-shot examples defined:

Example 1:
  Document preview: Invoice #A1234
From: TechCo Inc.
To: Customer ABC
...
  Extracted: {'invoice_number': 'A1234', 'vendor_name': 'TechCo Inc.', 'customer_name': 'Customer ABC', 'total': 100.0}

Example 2:
  Document preview: INVOICE
Number: INV-2024-99
Seller: MegaCorp LLC
B...
  Extracted: {'invoice_number': 'INV-2024-99', 'vendor_name': 'MegaCorp LLC', 'customer_name': 'SmallBiz Co.', 'total': 5400.0}


In [12]:
def few_shot_extract(document: str, examples: list[dict]) -> dict:
    """
    Extract data using few-shot examples.
    
    Args:
        document: The document to extract from
        examples: List of {'document': str, 'extracted': dict} examples
        
    Returns:
        Extracted data as a dictionary
    """
    # Build messages with examples
    messages = [
        {
            'role': 'system',
            'content': '''You are a document data extraction assistant.
Extract structured data from documents and return as JSON.
Follow the exact format shown in the examples.'''
        }
    ]
    
    # Add few-shot examples as conversation turns
    for ex in examples:
        messages.append({
            'role': 'user',
            'content': f'Extract data from this document:\n{ex["document"]}'
        })
        messages.append({
            'role': 'assistant',
            'content': json.dumps(ex['extracted'])
        })
    
    # Add the actual document to extract
    messages.append({
        'role': 'user',
        'content': f'Extract data from this document:\n{document}'
    })
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
        temperature=0
    )
    
    # Parse the response
    result_text = response.choices[0].message.content
    return json.loads(result_text)

# Test few-shot extraction
result = few_shot_extract(sample_invoice, FEW_SHOT_EXAMPLES)

print('📤 Few-shot extraction result:')
print(json.dumps(result, indent=2))

📤 Few-shot extraction result:
{
  "invoice_number": "INV-2024-0842",
  "vendor_name": "TechCorp Solutions Inc.",
  "customer_name": "Acme Corporation",
  "total": 24686.46
}


---
## 5. Compare: Zero-Shot vs Few-Shot

Let's test both approaches on multiple invoices and compare results.

In [13]:
# Test on all invoices
results = []

for name, content in list(INVOICES.items())[:4]:  # Test on first 4
    print(f'\n📄 Processing: {name}')
    
    # Zero-shot
    try:
        zero_result = zero_shot_with_system(content, fields_to_extract)
        zero_parsed = json.loads(zero_result)
        zero_status = '✅'
    except Exception as e:
        zero_parsed = {'error': str(e)}
        zero_status = '❌'
    
    # Few-shot
    try:
        few_parsed = few_shot_extract(content, FEW_SHOT_EXAMPLES)
        few_status = '✅'
    except Exception as e:
        few_parsed = {'error': str(e)}
        few_status = '❌'
    
    results.append({
        'document': name,
        'zero_shot': zero_parsed,
        'zero_status': zero_status,
        'few_shot': few_parsed,
        'few_status': few_status
    })
    
    print(f'  Zero-shot {zero_status}: {zero_parsed.get("vendor_name", "N/A")}')
    print(f'  Few-shot {few_status}: {few_parsed.get("vendor_name", "N/A")}')


📄 Processing: invoice_001_corporate
  Zero-shot ✅: TechCorp Solutions Inc.
  Few-shot ✅: TechCorp Solutions Inc.

📄 Processing: invoice_002_simple_receipt
  Zero-shot ✅: CloudHost Services
  Few-shot ✅: CloudHost Services

📄 Processing: invoice_003_email_style
  Zero-shot ✅: DesignPro Studio
  Few-shot ✅: DesignPro Studio LLC

📄 Processing: invoice_004_european_format
  Zero-shot ✅: Müller & Schmidt GmbH
  Few-shot ✅: Müller & Schmidt GmbH


In [14]:
# Summary comparison
md('''## Comparison Summary

| Document | Zero-Shot | Few-Shot |
|----------|:---------:|:--------:|''')

for r in results:
    print(f"| {r['document'][:25]} | {r['zero_status']} | {r['few_status']} |")

## Comparison Summary

| Document | Zero-Shot | Few-Shot |
|----------|:---------:|:--------:|

| invoice_001_corporate | ✅ | ✅ |
| invoice_002_simple_receip | ✅ | ✅ |
| invoice_003_email_style | ✅ | ✅ |
| invoice_004_european_form | ✅ | ✅ |


---
## 6. Extracting More Fields

Let's extract a complete invoice with all fields.

In [15]:
def extract_full_invoice(document: str) -> dict:
    """
    Extract complete invoice data with all fields.
    """
    system_prompt = '''You are an invoice data extraction specialist.
Extract all relevant fields from the invoice and return as JSON.

Required fields:
- invoice_number: The invoice ID/number
- invoice_date: Date the invoice was issued
- due_date: Payment due date (null if not specified)
- vendor_name: Company issuing the invoice
- vendor_address: Vendor's address (null if not found)
- customer_name: Company/person being billed
- line_items: Array of {description, quantity, unit_price, amount}
- subtotal: Sum before tax
- tax_amount: Tax in dollars (null if not specified)
- total: Final amount due
- payment_status: Paid/Pending/Overdue (null if not specified)

Return ONLY valid JSON. Use null for missing fields.'''

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'Extract from this invoice:\n{document}'}
        ],
        temperature=0
    )
    
    return json.loads(response.choices[0].message.content)

# Test full extraction
full_result = extract_full_invoice(sample_invoice)

print('📤 Full invoice extraction:')
print(json.dumps(full_result, indent=2))

📤 Full invoice extraction:
{
  "invoice_number": "INV-2024-0842",
  "invoice_date": "2024-01-15",
  "due_date": "2024-02-14",
  "vendor_name": "TechCorp Solutions Inc.",
  "vendor_address": "1234 Innovation Drive, Suite 500, San Francisco, CA 94105",
  "customer_name": "Acme Corporation",
  "line_items": [
    {
      "description": "Cloud Infrastructure Setup",
      "quantity": 1,
      "unit_price": 8500.0,
      "amount": 8500.0
    },
    {
      "description": "Custom API Development",
      "quantity": 3,
      "unit_price": 2200.0,
      "amount": 6600.0
    },
    {
      "description": "Database Migration Services",
      "quantity": 1,
      "unit_price": 3750.0,
      "amount": 3750.0
    },
    {
      "description": "Security Audit & Compliance",
      "quantity": 1,
      "unit_price": 4200.0,
      "amount": 4200.0
    },
    {
      "description": "Training Sessions (4 hours)",
      "quantity": 2,
      "unit_price": 450.0,
      "amount": 900.0
    }
  ],
  "subtotal

## 🎯 Summary

### Key Takeaways

1. **Zero-shot extraction** — simple to implement but produces inconsistent output formats
2. **Few-shot learning** — providing examples dramatically improves format consistency and field accuracy
3. **System prompts** — define extraction behavior and output format requirements clearly
4. **Temperature=0** — ensures deterministic extraction for consistent results
5. **JSON parsing** — always validate that LLM output is valid JSON before using it

---
## Next Steps

In the next notebook, we'll learn how to:
- Use **JSON mode** for guaranteed valid JSON
- Define **Pydantic schemas** for type validation
- Handle **nested structures** and optional fields

→ Continue to **mini-doc-extractor-02-schema-validation.ipynb**